# Phase 1 Validation Notebook (`eventlog/preparation.py`)

This notebook validates Phase 1 helpers (`1.1`–`1.12`), **case lifecycle columns (`1.14`–`1.18`)** — `activity_occurrence_index`, `activity_position_ratio`, `case_throughput_time`, `time_since_case_start`, `remaining_case_time` — plus **`detect_concurrency`**, **`make_chronological_window`**, and optional **`chronological_window`** injection, using `create_sample_event_log`.

Run cells top-to-bottom in Databricks or a local PySpark notebook with an active `spark` session.

In [25]:
import sys
!{sys.executable} -m pip install pyspark==3.5.0

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [26]:
import sys
from pathlib import Path

from pyspark.sql import functions as F

try:
    import pyspark  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PySpark is not installed in this Python environment. "
        "Install it with: pip install pyspark==3.5.0, then restart the kernel/session."
    ) from exc

# Make imports robust when notebook is run from repo root or dev/.
candidate_roots = [Path.cwd(), Path.cwd().parent]
for root in candidate_roots:
    dev_file = root / "dev" / "sample_data.py"
    pkg_dir = root / "pm_spark"
    if dev_file.exists() and pkg_dir.exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        if str(root / "dev") not in sys.path:
            sys.path.insert(0, str(root / "dev"))
        break

try:
    from dev.sample_data import create_sample_event_log
except ModuleNotFoundError:
    from sample_data import create_sample_event_log

import importlib
import pm_spark.eventlog.preparation as preparation

preparation = importlib.reload(preparation)

activity_sequence_number = preparation.activity_sequence_number
case_start_timestamp = preparation.case_start_timestamp
case_end_timestamp = preparation.case_end_timestamp
is_first_activity_flag = preparation.is_first_activity_flag
is_last_activity_flag = preparation.is_last_activity_flag
first_occurrence_of_activity = preparation.first_occurrence_of_activity
last_occurrence_of_activity = preparation.last_occurrence_of_activity
duplicate_activity_flag = preparation.duplicate_activity_flag
missing_activity_flag = preparation.missing_activity_flag
event_count_per_case = preparation.event_count_per_case
carry_forward_dimension_value = preparation.carry_forward_dimension_value
first_nonnull_in_dimension = preparation.first_nonnull_in_dimension
last_nonnull_in_dimension = preparation.last_nonnull_in_dimension
time_since_previous_event = preparation.time_since_previous_event
previous_activity = preparation.previous_activity
next_activity = preparation.next_activity
detect_concurrency = preparation.detect_concurrency
make_chronological_window = preparation.make_chronological_window
activity_occurrence_index = preparation.activity_occurrence_index
activity_position_ratio = preparation.activity_position_ratio
case_throughput_time = preparation.case_throughput_time
time_since_case_start = preparation.time_since_case_start
remaining_case_time = preparation.remaining_case_time

TARGET_ACTIVITY = "B"
DIMENSION_COL = "resource"

In [27]:
from pyspark.sql import SparkSession

# Databricks: reuse the attached cluster session. Local notebook: build one.
spark = SparkSession.getActiveSession()
if spark is None:
    spark = (
        SparkSession.builder.appName("pm_spark_phase1_validation")
        .master("local[*]")
        .getOrCreate()
    )

26/03/22 13:08:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/22 13:08:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [28]:
df = create_sample_event_log(spark) 
print("Input rows:", df.count())
df.orderBy("case_key", "timestamp", "activity").show(100, truncate=False)

Input rows: 31
+--------+-------------+-------------------+--------+----------+
|case_key|activity     |timestamp          |resource|department|
+--------+-------------+-------------------+--------+----------+
|C001    |A            |2026-03-18 09:00:00|R1      |D1        |
|C001    |B            |2026-03-18 09:05:00|R1      |D1        |
|C001    |C            |2026-03-18 09:10:00|R1      |D1        |
|C001    |D            |2026-03-18 09:20:00|R1      |D1        |
|C002    |A            |2026-03-18 09:00:00|R1      |D1        |
|C002    |B            |2026-03-18 09:04:00|R1      |D1        |
|C002    |Manual_Review|2026-03-18 09:08:00|R1      |D1        |
|C002    |C            |2026-03-18 09:15:00|R2      |D2        |
|C002    |D            |2026-03-18 09:25:00|R2      |D2        |
|C003    |A            |2026-03-18 10:00:00|NULL    |D1        |
|C003    |C            |2026-03-18 10:10:00|R3      |NULL      |
|C003    |D            |2026-03-18 10:30:00|R3      |D3        |
|C004    |

### Concurrency (`detect_concurrency`)

Adds **`is_concurrent`** (true when more than one row shares the same case and
timestamp) and **`concurrent_activities`**: for concurrent instants, all
activities at that moment are collected into one **lexicographically sorted**
array so `{B, P}` and `{P, B}` both become `["B", "P"]`. Otherwise the column
is a single-element array for that row’s activity.

The sample log’s **C005** has two events at **`2026-03-19 09:05:00`** (`B` and
`P`)—both rows should show `is_concurrent = true` and the same block array.

In [29]:
df_conc = detect_concurrency(
    df,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
)
df_conc.filter(F.col("case_key") == "C005").select(
    "case_key",
    "activity",
    "timestamp",
    "is_concurrent",
    "concurrent_activities",
).orderBy("timestamp", "activity").show(truncate=False)

# Expect: both rows at 2026-03-19 09:05:00 have is_concurrent=True and the same
# concurrent_activities array (sorted): ["B", "P"].

+--------+--------+-------------------+-------------+---------------------+
|case_key|activity|timestamp          |is_concurrent|concurrent_activities|
+--------+--------+-------------------+-------------+---------------------+
|C005    |A       |2026-03-19 09:00:00|false        |[A]                  |
|C005    |B       |2026-03-19 09:05:00|true         |[B, P]               |
|C005    |P       |2026-03-19 09:05:00|true         |[B, P]               |
|C005    |C       |2026-03-19 09:15:00|false        |[C]                  |
|C005    |D       |2026-03-19 09:20:00|false        |[D]                  |
+--------+--------+-------------------+-------------+---------------------+



### Shared chronological window (`make_chronological_window` + `chronological_window`)

Build one `Window` and pass it into lag/lead helpers so ordering stays aligned. Below we add a synthetic `event_row_id` tie-breaker (recommended when you inject a window; see `make_chronological_window` docstring).

In [30]:
df_eid = df.withColumn("event_row_id", F.monotonically_increasing_id())
chron_w = make_chronological_window(
    "case_key",
    "activity",
    "timestamp",
    "event_row_id",
)
df_inj = previous_activity(
    df_eid,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    chronological_window=chron_w,
)
df_inj = next_activity(
    df_inj,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    chronological_window=chron_w,
)
df_inj.filter(F.col("case_key") == "C001").select(
    "activity",
    "timestamp",
    "previous_activity",
    "next_activity",
).orderBy("timestamp", "activity").show(truncate=False)

# Expect: matches the main pipeline for C001 (first prev null, last next null).

+--------+-------------------+-----------------+-------------+
|activity|timestamp          |previous_activity|next_activity|
+--------+-------------------+-----------------+-------------+
|A       |2026-03-18 09:00:00|NULL             |B            |
|B       |2026-03-18 09:05:00|A                |C            |
|C       |2026-03-18 09:10:00|B                |D            |
|D       |2026-03-18 09:20:00|C                |NULL         |
+--------+-------------------+-----------------+-------------+



In [31]:
df_phase1 = activity_sequence_number(
    df=df,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="activity_index",
)

df_phase1 = activity_occurrence_index(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="activity_occurrence_index",
)

df_phase1 = activity_position_ratio(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="activity_position_ratio",
)

df_phase1 = case_start_timestamp(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="case_start_ts",
)

df_phase1 = case_end_timestamp(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="case_end_ts",
)

df_phase1 = case_throughput_time(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="case_throughput_time_us",
)

df_phase1 = time_since_case_start(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="time_since_case_start_us",
)

df_phase1 = remaining_case_time(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="remaining_case_time_us",
)

df_phase1 = is_first_activity_flag(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="is_first_activity",
)

df_phase1 = is_last_activity_flag(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="is_last_activity",
)

df_phase1 = first_occurrence_of_activity(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    activity_name=TARGET_ACTIVITY,
    output_col="first_b_ts",
)

df_phase1 = last_occurrence_of_activity(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    activity_name=TARGET_ACTIVITY,
    output_col="last_b_ts",
)

df_phase1 = duplicate_activity_flag(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    activity_name=TARGET_ACTIVITY,
    output_col="has_duplicate_b",
)

df_phase1 = missing_activity_flag(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    expected_activity=TARGET_ACTIVITY,
    output_col="is_missing_b",
)

df_phase1 = event_count_per_case(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="event_count_per_case",
)

df_phase1 = carry_forward_dimension_value(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col=DIMENSION_COL,
    output_col="resource_filled",
)

df_phase1 = first_nonnull_in_dimension(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col=DIMENSION_COL,
    output_col="resource_first_nonnull",
)

df_phase1 = last_nonnull_in_dimension(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    dimension_col=DIMENSION_COL,
    output_col="resource_last_nonnull",
)

df_phase1 = time_since_previous_event(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="microseconds_since_previous_event",
)

df_phase1 = previous_activity(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="previous_activity",
)

df_phase1 = next_activity(
    df=df_phase1,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="next_activity",
)

result = df_phase1.orderBy("case_key", "timestamp", "activity")
result.show(200, truncate=False)

+--------+-------------+-------------------+--------+----------+--------------+-------------------------+-----------------------+-------------------+-------------------+-----------------------+------------------------+----------------------+-----------------+----------------+-------------------+-------------------+---------------+------------+--------------------+---------------+----------------------+---------------------+---------------------------------+-----------------+-------------+
|case_key|activity     |timestamp          |resource|department|activity_index|activity_occurrence_index|activity_position_ratio|case_start_ts      |case_end_ts        |case_throughput_time_us|time_since_case_start_us|remaining_case_time_us|is_first_activity|is_last_activity|first_b_ts         |last_b_ts          |has_duplicate_b|is_missing_b|event_count_per_case|resource_filled|resource_first_nonnull|resource_last_nonnull|microseconds_since_previous_event|previous_activity|next_activity|
+--------+--

In [32]:
result.select(
    "case_key",
    "activity",
    "timestamp",
    "activity_index",
    "activity_occurrence_index",
    "activity_position_ratio",
    "case_start_ts",
    "case_end_ts",
    "case_throughput_time_us",
    "time_since_case_start_us",
    "remaining_case_time_us",
    "is_first_activity",
    "is_last_activity",
    "first_b_ts",
    "last_b_ts",
    "has_duplicate_b",
    "is_missing_b",
    "event_count_per_case",
    "resource",
    "resource_filled",
    "resource_first_nonnull",
    "resource_last_nonnull",
    "microseconds_since_previous_event",
    "previous_activity",
    "next_activity",
).orderBy("case_key", "timestamp", "activity").show(200, truncate=False)

+--------+-------------+-------------------+--------------+-------------------------+-----------------------+-------------------+-------------------+-----------------------+------------------------+----------------------+-----------------+----------------+-------------------+-------------------+---------------+------------+--------------------+--------+---------------+----------------------+---------------------+---------------------------------+-----------------+-------------+
|case_key|activity     |timestamp          |activity_index|activity_occurrence_index|activity_position_ratio|case_start_ts      |case_end_ts        |case_throughput_time_us|time_since_case_start_us|remaining_case_time_us|is_first_activity|is_last_activity|first_b_ts         |last_b_ts          |has_duplicate_b|is_missing_b|event_count_per_case|resource|resource_filled|resource_first_nonnull|resource_last_nonnull|microseconds_since_previous_event|previous_activity|next_activity|
+--------+-------------+----------

In [33]:
case_summary = result.select(
    "case_key",
    "first_b_ts",
    "last_b_ts",
    "has_duplicate_b",
    "is_missing_b",
    "event_count_per_case",
    "resource_first_nonnull",
    "resource_last_nonnull",
).dropDuplicates(["case_key"]).orderBy("case_key")

case_summary.show(100, truncate=False)

result.filter(F.col("case_key") == "C004").select(
    "case_key",
    "activity",
    "timestamp",
    "microseconds_since_previous_event",
    "previous_activity",
    "next_activity",
).orderBy("timestamp", "activity").show(truncate=False)

+--------+-------------------+-------------------+---------------+------------+--------------------+----------------------+---------------------+
|case_key|first_b_ts         |last_b_ts          |has_duplicate_b|is_missing_b|event_count_per_case|resource_first_nonnull|resource_last_nonnull|
+--------+-------------------+-------------------+---------------+------------+--------------------+----------------------+---------------------+
|C001    |2026-03-18 09:05:00|2026-03-18 09:05:00|false          |false       |4                   |R1                    |R1                   |
|C002    |2026-03-18 09:04:00|2026-03-18 09:04:00|false          |false       |5                   |R1                    |R2                   |
|C003    |NULL               |NULL               |false          |true        |3                   |R3                    |R3                   |
|C004    |2026-03-18 11:05:00|2026-03-18 11:08:00|true           |false       |5                   |R1                    |R

## Expected checks

Use the `case_summary` output to verify:

- `C004` -> `has_duplicate_b = true`
- `C003` -> `is_missing_b = true`, `first_b_ts = null`, `last_b_ts = null`
- `C001` -> `event_count_per_case = 4`
- `C002` -> `event_count_per_case = 5`
- `C003` -> `resource_first_nonnull = R3`, `resource_last_nonnull = R3`
- `C006` -> `resource_first_nonnull = R5`, `resource_last_nonnull = R6`
- `C007` -> `resource_first_nonnull = null`, `resource_last_nonnull = null`

For forward fill check (`1.8`):

- Filter to `case_key = 'C004'` and inspect timestamp `2026-03-18 11:08:00`.
- `resource` is null while `resource_filled` should be `R1`.

For new enrichment checks:

- `previous_activity`: first row per case is null.
- `next_activity`: last row per case is null.
- `time_since_previous_event`:
  - first row per case is null,
  - `C004` at `11:08` is `180000000` microseconds after previous event,
  - `C004` at `11:18` is `600000000` microseconds after previous event.

For `detect_concurrency` (cells above):

- `C005` at `2026-03-19 09:05:00`: both rows `is_concurrent = true`, identical `concurrent_activities` = `["B", "P"]` (sorted).

For injected `chronological_window`:

- `C001` prev/next chain should match the main pipeline when `event_row_id` is included in `make_chronological_window`.

For case lifecycle columns (`1.14`–`1.18`):

- `activity_occurrence_index`: per activity label within a case — `C004` first `B` → `1`, second `B` → `2`; `C006` second `A` → `2`. No `chronological_window` argument: pass the same tie-break column as `event_order_col` when you need ordering aligned with `make_chronological_window`.
- `activity_position_ratio`: `C001` four events → `0.0`, `0.333…`, `0.666…`, `1.0`; single-event cases would be `0.0` (none in sample).
- `case_throughput_time_us`: constant per case; equals `time_since_case_start_us + remaining_case_time_us` on each row (microsecond alignment with `microseconds_since_previous_event`).
- `time_since_case_start_us`: first event per case → `0`.
- `remaining_case_time_us`: last event per case → `0`.

In [34]:
spark.stop()